<a href="https://colab.research.google.com/github/rix031110/LLM_implementation/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
## 1. Setup & Dependencies

In [9]:
# Install required packages
!pip install -q transformers datasets faiss-cpu sentence-transformers torch numpy pandas matplotlib seaborn scikit-learn pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 14.8 MB/s eta 0:00:00


In [10]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Tuple
import time
import warnings
import os
warnings.filterwarnings('ignore')

from transformers import (
    AutoTokenizer,
    AutoModel,
    GPT2LMHeadModel,
    GPT2Tokenizer,
    set_seed
)
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
import faiss
from sklearn.metrics.pairwise import cosine_similarity
from pypdf import PdfReader

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [11]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
# Define the path to your uploaded PDF file
pdf_file_path = '/content/drive/MyDrive/scouting for boys.pdf'

# Check if the file exists
if not os.path.exists(pdf_file_path):
    print(f"Error: The file '{pdf_file_path}' was not found. Please ensure you have uploaded it correctly.")
else:
    # Create a PdfReader object
    reader = PdfReader(pdf_file_path)

    # Get the number of pages
    num_pages = len(reader.pages)
    print(f"The PDF has {num_pages} page(s).")

    # Extract text from each page
    extracted_text = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        if text:
            extracted_text.append(f"--- Page {i+1} ---\n{text}")
        else:
            extracted_text.append(f"--- Page {i+1} ---\n[No text found on this page]")

    # Print the first 500 characters of the extracted text
    full_text = "\n".join(extracted_text)
    print("\n--- Extracted Text (first 500 chars) ---")
    print(full_text[:800])

    # You can also store the full text in a variable for further processing
    # For example, to integrate with a RAG pipeline:
    # document_for_rag = {'id': 'pdf_doc_1', 'title': 'Sample PDF Content', 'text': full_text}
    # print(document_for_rag)

The PDF has 223 page(s).

--- Extracted Text (first 500 chars) ---
--- Page 1 ---
 
Scouting For Boys 
A Handbook for Instruction in Good 
Citizenship Through Woodcraft 
 
By 
LORD BADEN-POWELL OF GILWELL 
Founder of the Boy Scout Movement 
 
 
 
 
 
--- Page 2 ---
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Editor’s Note: 
 
The reader is reminded that these texts have been written a long time ago.  Consequently, they 
may use some terms or use expressions which were current at the time, regardless of what we  
may think of them at the beginning of the 21 st century. For reasons of historical accuracy they 
have been preserved in their original form. 
 
If you find them offensive, we ask you to please delete this file from your system. 
This and other traditional Scouting texts may be downloaded from the Dump. 
 
Downloaded from: 
“The Dump” at Scoutscan.com 
http://www


---
## 3. Building the Knowledge Base

First, we need a collection of documents to retrieve from. We'll use a sample dataset and create a simple knowledge base.

In [13]:
basic_knowledge = []

# Split full_text into individual pages
page_contents = full_text.split('--- Page ')[1:]  # Skip the first empty split

for i, page_content in enumerate(page_contents):
    page_number_str, text_content = page_content.split(' ---\n', 1)
    page_number = int(page_number_str.strip())

    basic_knowledge.append({
        'id': page_number,
        'title': f'Scouting for boys {page_number}',
        'text': text_content.strip()
    })

# You can print the first few entries to verify
print(basic_knowledge[100]) # if there's a second page

{'id': 101, 'title': 'Scouting for boys 101', 'text': 'Cooking Hints \nWhen boiling a pot of water on the fire, do not jam the lid on too firmly. When the steam forms inside the \npot, it must have some means of escape. To find out when the water is beginning to boil, you need not \ntake off the lid and look, but just hold the end of a stick or knife to the pot, and if the water is boiling you \nwill feel the pot trembling. \nOatmeal Porridge—Pour into a pot one cup of water for each person. Add a pinch of salt for each cup. \nWhen the water boils, sprinkle oatmeal in it while stirring with a stick or large spoon. The amount of \noatmeal depends upon whether you want the porridge thick or thin. Simmer the porridge until it is done, \nstirring all the time. \nDon’t do as I did once when I was a tenderfoot. It was my turn to cook, so I thought I would vary the \ndinner by giving them soup. I had some pea-flour, and I mixed it with water and boiled it up, and served it \nas pea-soup. But 

In [15]:
df_kb = pd.DataFrame(basic_knowledge)
print(f"Knowledge Base: {len(df_kb)} documents\n")
print(df_kb[['id', 'title']].head(10).to_string(index=False))

Knowledge Base: 223 documents

 id                title
  1  Scouting for boys 1
  2  Scouting for boys 2
  3  Scouting for boys 3
  4  Scouting for boys 4
  5  Scouting for boys 5
  6  Scouting for boys 6
  7  Scouting for boys 7
  8  Scouting for boys 8
  9  Scouting for boys 9
 10 Scouting for boys 10


---
## 4. Step 1: RETRIEVAL - Building the Vector Database

To retrieve relevant documents, we need to:
1. Convert documents to **embeddings** (dense vectors)
2. Store embeddings in a **vector database**
3. Convert queries to embeddings
4. Find most similar documents using **semantic search**

### 4.1 Create Document Embeddings

We'll use **Sentence-BERT** to create high-quality embeddings.

In [16]:
# Load embedding model (Sentence-BERT)
print("Loading embedding model...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')  # Fast and efficient
embedding_dim = embedding_model.get_sentence_embedding_dimension()
print(f"Embedding model loaded. Dimension: {embedding_dim}")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded. Dimension: 384


In [17]:
# Generate embeddings for all documents
print("\nGenerating embeddings for documents...")
start_time = time.time()

# Combine title and text for richer embeddings
df_kb['combined_text'] = df_kb['title'] + ': ' + df_kb['text']
documents = df_kb['combined_text'].tolist()

# Encode all documents
doc_embeddings = embedding_model.encode(documents, show_progress_bar=True)
doc_embeddings = np.array(doc_embeddings).astype('float32')

embedding_time = time.time() - start_time
print(f"\nEmbeddings generated in {embedding_time:.2f} seconds")
print(f"Embeddings shape: {doc_embeddings.shape}")


Generating embeddings for documents...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]


Embeddings generated in 32.14 seconds
Embeddings shape: (223, 384)


### 4.2 Build FAISS Index

**FAISS** (Facebook AI Similarity Search) is a library for efficient similarity search in high-dimensional spaces.

In [18]:
# Create FAISS index
print("Building FAISS index...")
index = faiss.IndexFlatL2(embedding_dim)  # L2 distance (Euclidean)

# Normalize vectors for cosine similarity
faiss.normalize_L2(doc_embeddings)
index.add(doc_embeddings)

print(f"FAISS index built with {index.ntotal} vectors")

Building FAISS index...
FAISS index built with 223 vectors


### 4.3 Implement Retrieval Function

In [19]:
def retrieve_documents(query: str, k: int = 3) -> List[Dict]:
    """
    Retrieve top-k most relevant documents for a query.

    Args:
        query: The search query
        k: Number of documents to retrieve

    Returns:
        List of retrieved documents with scores
    """
    # Encode query
    query_embedding = embedding_model.encode([query])
    query_embedding = np.array(query_embedding).astype('float32')
    faiss.normalize_L2(query_embedding)

    # Search in FAISS index
    distances, indices = index.search(query_embedding, k)

    #distances: A 2D array of shape (1, k) — the similarity/distance scores for each returned neighbor.
    #indices: A 2D array of shape (1, k) — the positions of the nearest neighbors in the original index.

    # Convert distances to similarity scores (cosine similarity)
    similarities = 1 - distances[0]

    # Retrieve documents
    results = []
    for idx, score in zip(indices[0], similarities):
        doc = df_kb.iloc[idx].to_dict()
        doc['retrieval_score'] = float(score)
        results.append(doc)

    return results

### 4.4 Test Retrieval

In [20]:
# Test queries
test_queries = [
    "Who wrote scouting for boys"
]

print("Testing Retrieval System:\n")
print("="*80)

for query in test_queries:
    print(f"\nQuery: '{query}'\n")
    retrieved = retrieve_documents(query, k=3)

    for i, doc in enumerate(retrieved, 1):
        print(f"{i}. {doc['title']} (score: {doc['retrieval_score']:.3f})")
        print(f"   {doc['text'][:100]}...\n")
    print("="*80)

Testing Retrieval System:


Query: 'Who wrote scouting for boys'

1. Scouting for boys 70 (score: 0.422)
   ...

2. Scouting for boys 71 (score: 0.389)
   ...

3. Scouting for boys 1 (score: 0.363)
   Scouting For Boys 
A Handbook for Instruction in Good 
Citizenship Through Woodcraft 
 
By 
LORD BAD...



---
## 5. Step 2: AUGMENTATION - Building the Prompt

Now we combine the query with retrieved context to create an augmented prompt.

In [30]:
def create_augmented_prompt(query: str, retrieved_docs: List[Dict], max_context_length: int = 900) -> str:
    """
    Create an augmented prompt by combining query with retrieved context.

    Args:
        query: User's question
        retrieved_docs: List of retrieved documents
        max_context_length: Maximum characters for context

    Returns:
        Augmented prompt string
    """
    # Build context from retrieved documents
    context_parts = []
    current_length = 0

    for doc in retrieved_docs:
        doc_text = f"[{doc['title']}] {doc['text']}"
        if current_length + len(doc_text) <= max_context_length:
            context_parts.append(doc_text)
            current_length += len(doc_text)
        else:
            break

    context = "\n\n".join(context_parts)

    # Create prompt template
    prompt = f"""Context:
{context}

Question: {query}

Answer based on the context above:"""

    return prompt

In [31]:
# Example augmented prompt
query = "What is a scout?"
retrieved = retrieve_documents(query, k=2)
augmented_prompt = create_augmented_prompt(query, retrieved)

print("Example Augmented Prompt:\n")
print("="*80)
print(augmented_prompt)
print("="*80)

Example Augmented Prompt:

Context:


Question: What is a scout?

Answer based on the context above:


---
## 6. Step 3: GENERATION - Producing the Answer

Now we feed the augmented prompt to a language model to generate an answer.

In [32]:
# Load generation model (GPT-2)
print("Loading generation model...")
gen_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
gen_model = GPT2LMHeadModel.from_pretrained('gpt2').to(device)
gen_tokenizer.pad_token = gen_tokenizer.eos_token
print("Generation model loaded (GPT-2)")

Loading generation model...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generation model loaded (GPT-2)


In [33]:
def generate_answer(prompt: str, max_length: int = 800) -> str:
    """
    Generate an answer using the language model.

    Args:
        prompt: The augmented prompt with context
        max_length: Maximum length of generated answer

    Returns:
        Generated answer string
    """
    input_ids = gen_tokenizer.encode(prompt, return_tensors='pt').to(device)

    # Generate with controlled parameters
    output = gen_model.generate(
        input_ids,
        max_length=input_ids.shape[1] + max_length,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        num_return_sequences=1,
        pad_token_id=gen_tokenizer.eos_token_id
    )

    # Decode and extract only the new generated part
    full_text = gen_tokenizer.decode(output[0], skip_special_tokens=True)

    # Extract answer (everything after the prompt)
    if "Answer based on the context above:" in full_text:
        answer = full_text.split("Answer based on the context above:")[-1].strip()
    else:
        answer = full_text[len(prompt):].strip()

    return answer

---
## 7. Complete RAG Pipeline

In [34]:
def rag_pipeline(query: str, k: int = 3, verbose: bool = True) -> Dict:
    """
    Complete RAG pipeline: Retrieve → Augment → Generate

    Args:
        query: User's question
        k: Number of documents to retrieve
        verbose: Print intermediate steps

    Returns:
        Dictionary with answer, retrieved docs, and metadata
    """
    start_time = time.time()

    # Step 1: Retrieve
    if verbose:
        print(f"Query: '{query}'\n")
        print("[1/3] Retrieving relevant documents...")

    retrieval_start = time.time()
    retrieved_docs = retrieve_documents(query, k=k)
    retrieval_time = time.time() - retrieval_start

    if verbose:
        print(f"Retrieved {len(retrieved_docs)} documents in {retrieval_time:.3f}s")
        for i, doc in enumerate(retrieved_docs, 1):
            print(f"  {i}. {doc['title']} (relevance: {doc['retrieval_score']:.3f})")

    # Step 2: Augment
    if verbose:
        print("\n[2/3] Creating augmented prompt...")

    augment_start = time.time()
    augmented_prompt = create_augmented_prompt(query, retrieved_docs)
    augment_time = time.time() - augment_start

    if verbose:
        print(f"Augmentation completed in {augment_time:.3f}s")

    # Step 3: Generate
    if verbose:
        print("\n[3/3] Generating answer...")

    gen_start = time.time()
    answer = generate_answer(augmented_prompt)
    gen_time = time.time() - gen_start

    if verbose:
        print(f"Answer generated in {gen_time:.3f}s\n")

    total_time = time.time() - start_time

    return {
        'query': query,
        'answer': answer,
        'retrieved_docs': retrieved_docs,
        'augmented_prompt': augmented_prompt,
        'timings': {
            'retrieval': retrieval_time,
            'augmentation': augment_time,
            'generation': gen_time,
            'total': total_time
        }
    }

---
## 8. Test the Complete RAG Pipeline

In [42]:
# Example queries to test the RAG pipeline
test_queries = [
    "what is a scout?"
]

print("\n" + "="*80)
print("RAG PIPELINE DEMO")
print("="*80)

for query in test_queries[:1]:  # Test with first query
    print(f"\n\nProcessing: {query}\n")
    result = rag_pipeline(query, k=2, verbose=True)

    print("\n" + "-"*80)
    print("FINAL ANSWER:")
    print("-"*80)
    print(result['answer'])
    print("-"*80)
    print(f"\nTotal time: {result['timings']['total']:.3f} seconds")


RAG PIPELINE DEMO


Processing: what is a scout?

Query: 'what is a scout?'

[1/3] Retrieving relevant documents...
Retrieved 2 documents in 0.030s
  1. Scouting for boys 28 (relevance: 0.344)
  2. Scouting for boys 47 (relevance: 0.302)

[2/3] Creating augmented prompt...
Augmentation completed in 0.000s

[3/3] Generating answer...


KeyboardInterrupt: 